# old_rotation_tiers_expanded → training_stars pipeline

End-to-end pipeline for ingesting `cf_data/old_rotation_tiers_expanded.csv` into `training_stars.csv`.

**Steps:**
1. Parse Gaia DR3 source IDs — extract directly from `star_id` where possible, SIMBAD fallback for KIC/GJ/HD/named stars
2. Fetch Gaia DR3 astrometry + photometry from ARI Heidelberg TAP
3. Dereddening — Edenhofer 2023 + Bayestar 2019 fallback → A_Ks, A_Ks_err, bp_rp_0
4. 2MASS crossmatch — ARI TAP `tmass_psc_xsc_best_neighbour` → CDS TAPVizieR, SIMBAD fallback
5. Mann 2019 Mass-Ks relation — compute k_m_0, M_Ks, mass + asymmetric uncertainties
6. Gossage 2024 τ_cE interpolation → Rossby number
7. Filter to valid M dwarfs (4.5 < M_Ks < 10.5), map to training_stars schema, append

**Valid Mann 2019 range:** 4.5 < M_Ks < 10.5 (≈ 0.075–0.70 M☉)

**Expected M dwarfs to pass:** ~6 ASAS old-disk GJ stars + ~4 Chiti 2024 WD+MS M companions

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import sys
import time
import requests
import pkg_resources
from scipy.interpolate import interp1d
from astroquery.simbad import Simbad
from astropy.io import fits
import astropy.units as u
from astropy.coordinates import SkyCoord

from dustmaps.config import config
from dustmaps.bayestar import BayestarQuery
from dustmaps.edenhofer2023 import Edenhofer2023Query

sys.path.insert(0, 'Mann_2019')
import mk_mass

CF_DATA  = Path('cf_data')
ARI_SYNC = 'https://gaia.ari.uni-heidelberg.de/tap/sync'
CDS_TAP  = 'http://tapvizier.cds.unistra.fr/TAPVizieR/tap/sync'

# Must match dereddening.ipynb
config['data_dir'] = '/opt/anaconda3/lib/python3.12/site-packages/dustmaps/data'

def chunk_list(lst, n=500):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

print('Setup complete.')

Setup complete.


## Step 1 — Parse Gaia DR3 source IDs

In [2]:
src = pd.read_csv(CF_DATA / 'old_rotation_tiers_expanded.csv')
print(f'Loaded {len(src)} stars from old_rotation_tiers_expanded.csv')
print(f'Columns: {list(src.columns)}')
print()
print('Sample names:')
print(src['star_id'].tolist())

Loaded 117 stars from old_rotation_tiers_expanded.csv
Columns: ['star_id', 'alternate_id', 'prot_days', 'prot_err_days', 'prot_err_plus_days', 'prot_err_minus_days', 'age_gyr', 'age_err_gyr', 'age_err_plus_gyr', 'age_err_minus_gyr', 'bibcode', 'sample_name', 'tier_name', 'quality_note', 'teff_k', 'spectral_type_estimate', 'age_label_type', 'period_measurement_type', 'source_url']

Sample names:
['KIC 4914423', 'KIC 6116048', 'KIC 9410862', 'KIC 7106245', 'KIC 4914923', 'KIC 6933899', 'KIC 6521045', 'KIC 3544595', 'KIC 10516096', 'KIC 11401755', 'KIC 12069449', 'KIC 7296438', 'KIC 11295426', 'KIC 12069424', 'KIC 9955598', 'KIC 7680114', 'KIC 10586004', 'KIC 10514430', 'KIC 9098294', 'KIC 7871531', 'KIC 3656476', 'KIC 5950854', 'KIC 8424992', 'KIC 11772920', 'KIC 7970740', 'KIC 11904151', 'KIC 8349582', 'KIC 6278762', 'KIC 4143755', 'GJ 84', 'GJ 176', 'GJ 205', 'GJ 618A', 'GJ 411', 'GJ 699', 'Gaia EDR3 2051099015605500672', 'Gaia EDR3 2051104757981275264', 'Gaia EDR3 2051104891118311040'

In [3]:
# Extract numeric Gaia source ID from 'Gaia DR3 XXXX' or 'Gaia EDR3 XXXX' strings.
# EDR3 and DR3 share the same source catalog IDs.
# NOTE: must use pd.array(..., dtype="Int64") — NOT apply().astype("Int64").
# Mixing None with large ints in a plain Series causes pandas to infer float64,
# silently truncating 19-digit Gaia IDs (float64 only has ~15-16 sig figs).
def parse_gaia_id(star_id):
    m = re.match(r'Gaia (?:DR3|EDR3) (\d+)', str(star_id))
    return int(m.group(1)) if m else None

src['source_id'] = pd.array(
    [parse_gaia_id(s) for s in src['star_id']],
    dtype='Int64'
)

has_direct   = src['source_id'].notna()
needs_simbad = ~has_direct

print(f'Stars with Gaia ID in star_id:  {has_direct.sum()}')
print(f'Stars needing SIMBAD resolution: {needs_simbad.sum()}')
print()
print('Stars needing SIMBAD:')
for name in src.loc[needs_simbad, 'star_id']:
    print(f'  {name}')


Stars with Gaia ID in star_id:  67
Stars needing SIMBAD resolution: 50

Stars needing SIMBAD:
  KIC 4914423
  KIC 6116048
  KIC 9410862
  KIC 7106245
  KIC 4914923
  KIC 6933899
  KIC 6521045
  KIC 3544595
  KIC 10516096
  KIC 11401755
  KIC 12069449
  KIC 7296438
  KIC 11295426
  KIC 12069424
  KIC 9955598
  KIC 7680114
  KIC 10586004
  KIC 10514430
  KIC 9098294
  KIC 7871531
  KIC 3656476
  KIC 5950854
  KIC 8424992
  KIC 11772920
  KIC 7970740
  KIC 11904151
  KIC 8349582
  KIC 6278762
  KIC 4143755
  GJ 84
  GJ 176
  GJ 205
  GJ 618A
  GJ 411
  GJ 699
  KIC 6196457
  KIC 6521045
  KIC 8349582
  KIC 9955598
  KIC 10586004
  KIC 11401755
  16 Cyg A
  16 Cyg B
  KIC 3656476
  KIC 6116048
  KIC 7680114
  KIC 7871531
  KIC 9098294
  KIC 11244118
  HD 219134


In [4]:
# SIMBAD batch query to resolve common names (KIC, GJ, HD, named stars) to Gaia DR3 source_id.

simbad = Simbad()
simbad.add_votable_fields('ids')

def extract_gaia_dr3_id(ids_str):
    """Extract numeric Gaia DR3 source_id from SIMBAD ids string."""
    if not ids_str or str(ids_str) == 'None':
        return None
    for ident in str(ids_str).split('|'):
        m = re.match(r'Gaia DR3 (\d+)', ident.strip())
        if m:
            return int(m.group(1))
    return None

to_resolve = src[needs_simbad].copy()
query_names = to_resolve['star_id'].tolist()

print(f'Querying SIMBAD for {len(query_names)} stars...')
try:
    result = simbad.query_objects(query_names)
    resolved = 0
    for i, (idx, row) in enumerate(to_resolve.iterrows()):
        if i >= len(result):
            break
        gaia_id = extract_gaia_dr3_id(result['ids'][i] if result is not None else None)
        if gaia_id is not None:
            src.at[idx, 'source_id'] = gaia_id
            resolved += 1
        else:
            print(f'  No Gaia DR3 ID for: {row["star_id"]}')
    print(f'Resolved {resolved}/{len(query_names)} via SIMBAD')
except Exception as e:
    print(f'SIMBAD query failed: {e}')

print(f'\nTotal with source_id: {src["source_id"].notna().sum()}/{len(src)}')
unresolved = src[src['source_id'].isna()]
if len(unresolved) > 0:
    print(f'Unresolved: {unresolved["star_id"].tolist()}')

Querying SIMBAD for 50 stars...
Resolved 50/50 via SIMBAD

Total with source_id: 117/117


## Step 2 — Fetch Gaia DR3 astrometry & photometry

In [5]:
def query_gaia_astrometry(source_ids, retries=3, delay=20):
    """Fetch Gaia DR3 ra, dec, parallax, pmra, bp_rp, ruwe, G-mag from ARI sync TAP."""
    parts = []
    chunks = list(chunk_list(source_ids, n=500))
    for i, chunk in enumerate(chunks, start=1):
        print(f'  Chunk {i}/{len(chunks)}: {len(chunk)} IDs', end=' ', flush=True)
        ids_str = ','.join(str(x) for x in chunk)
        query = f"""
        SELECT source_id, ra, dec, parallax, parallax_error, pmra, pmdec,
               bp_rp, ruwe, phot_g_mean_mag, ebpminrp_gspphot
        FROM gaiadr3.gaia_source
        WHERE source_id IN ({ids_str})
        """
        for attempt in range(1, retries + 1):
            try:
                r = requests.post(ARI_SYNC,
                    data={'REQUEST': 'doQuery', 'LANG': 'ADQL', 'FORMAT': 'json', 'QUERY': query},
                    timeout=120)
                r.raise_for_status()
                d = r.json()
                cols = [c['name'] for c in d['metadata']]
                df = pd.DataFrame(d['data'], columns=cols)
                df['source_id'] = df['source_id'].astype('Int64')
                parts.append(df)
                print(f'→ {len(df)} rows')
                break
            except Exception as e:
                if attempt < retries:
                    print(f'\n    Attempt {attempt} failed ({e}), retrying in {delay}s...')
                    time.sleep(delay)
                else:
                    print(f'\n    All {retries} attempts failed — skipping chunk')
                    parts.append(pd.DataFrame(columns=[
                        'source_id','ra','dec','parallax','parallax_error',
                        'pmra','pmdec','bp_rp','ruwe','phot_g_mean_mag','ebpminrp_gspphot']))
        time.sleep(0.5)
    if not parts:
        return pd.DataFrame()
    return pd.concat(parts, ignore_index=True)

all_ids = src['source_id'].dropna().astype(int).unique().tolist()
print(f'Fetching Gaia DR3 astrometry for {len(all_ids)} unique source IDs...')
gaia_df = query_gaia_astrometry(all_ids)
print(f'\nRetrieved: {len(gaia_df)} / {len(all_ids)} stars')

Fetching Gaia DR3 astrometry for 105 unique source IDs...
  Chunk 1/1: 105 IDs → 105 rows

Retrieved: 105 / 105 stars


In [6]:
# Drop any stale Gaia columns before merging (safe to re-run)
gaia_cols = ['ra','dec','parallax','parallax_error','pmra','pmdec',
             'bp_rp','ruwe','phot_g_mean_mag','ebpminrp_gspphot']
src = src.drop(columns=[c for c in gaia_cols if c in src.columns])

src = src.merge(gaia_df, on='source_id', how='left')

print(f'Gaia data merged.')
print(f'  parallax available: {src["parallax"].notna().sum()}/{len(src)}')

no_gaia = src[src['parallax'].isna()]
if len(no_gaia) > 0:
    print(f'\nStars without Gaia parallax:')
    print(no_gaia[['star_id','source_id']].to_string())

Gaia data merged.
  parallax available: 117/117


## Step 3 — Dereddening

Same pipeline as `dereddening.ipynb`: Edenhofer 2023 (preferred) + Bayestar 2019 fallback → **A_Ks, A_Ks_err** (used for k_m_0 and the Mann mass computation).

`bp_rp_0`: primary = `bp_rp − ebpminrp_gspphot` (Gaia GSP-Phot); fallback = `bp_rp − A_BPminRP` (dust-map derived via Wang & Chen 2019, A_BPminRP/A_V = 0.413).

Conversion factors (Wang & Chen 2019, Table 3):
- E_Edenhofer × 0.829 → E(B-V)
- E_Bayestar × 0.88 → E(B-V)
- A_V = R_V × E(B-V), R_V = 3.1
- A_Ks/A_V = 0.078 ± 0.004
- A_BPminRP/A_V = 0.413

In [7]:
edenhofer = Edenhofer2023Query(load_samples=False, integrated=True)
bayestar   = BayestarQuery(max_samples=10)
print('Dust maps loaded.')

Integrating extinction map (this might take a couple of minutes)...
Optimizing map for querying (this might take a couple of seconds)...


Loading pixel_info ...
Loading samples ...
Loading best_fit ...
Replacing NaNs in reliable distance estimates ...
Sorting pixel_info ...
Extracting hp_idx_sorted and data_idx at each nside ...
  nside = 64
  nside = 128
  nside = 256
  nside = 512
  nside = 1024
t = 21.192 s
  pix_info:   0.267 s
   samples:  11.313 s
      best:   2.551 s
       nan:   0.026 s
      sort:   6.970 s
       idx:   0.065 s
Dust maps loaded.


In [8]:
Rv               = 3.1
A_Ks_div_AV      = 0.078
A_Ks_div_AV_err  = 0.004
A_BPminRP_div_AV = 0.413   # Wang & Chen 2019
NUM_DIST_SAMPLES  = 10

queryable = src[
    src['parallax'].notna() & (src['parallax'] > 0) &
    src['ra'].notna() & src['dec'].notna()
].copy()
queryable['dist_pc']     = 1000.0 / queryable['parallax']
queryable['dist_pc_err'] = np.where(
    queryable['parallax_error'].notna(),
    1000.0 * queryable['parallax_error'] / queryable['parallax']**2,
    0.0
)

print(f'Querying dust maps for {len(queryable)} stars...')
ext_rows = []

for i, (_, row) in enumerate(queryable.iterrows()):
    dist     = row['dist_pc']
    dist_err = row['dist_pc_err']

    if dist_err > 0:
        dist_samples = np.clip(np.random.normal(dist, dist_err, NUM_DIST_SAMPLES), 10.0, None)
    else:
        dist_samples = np.array([max(dist, 10.0)])

    e_samples, b_samples = [], []
    for d in dist_samples:
        coord = SkyCoord(row['ra'] * u.deg, row['dec'] * u.deg, distance=d * u.pc, frame='icrs')
        try:
            E_e = edenhofer(coord, mode='mean')
            if np.isfinite(E_e):
                e_samples.append(float(E_e))
        except Exception:
            pass
        try:
            E_b = bayestar(coord, mode='samples', return_flags=True)
            if E_b[1][0] & E_b[1][1]:
                b_samples.extend(E_b[0].tolist())
        except Exception:
            pass

    eden_valid = len(e_samples) > 0 and np.isfinite(np.nanmedian(e_samples))
    baye_valid = len(b_samples) > 0 and np.isfinite(np.nanmedian(b_samples))

    if eden_valid:
        EBV     = np.nanmedian(e_samples) * 0.829
        EBV_err = np.nanstd(e_samples)    * 0.829
        dustmap = 'Edenhofer'
    elif baye_valid:
        EBV     = np.nanmedian(b_samples) * 0.88
        EBV_err = np.nanstd(b_samples)    * 0.88
        dustmap = 'Bayestar'
    else:
        EBV = EBV_err = 0.0
        dustmap = 'none'

    A_V      = Rv * EBV
    A_V_err  = Rv * EBV_err
    A_Ks     = A_V * A_Ks_div_AV
    A_Ks_err = (A_Ks * np.sqrt((A_V_err / A_V)**2 + (A_Ks_div_AV_err / A_Ks_div_AV)**2)
                if A_V > 0 else 0.0)

    ext_rows.append({
        'source_id':    row['source_id'],
        'A_Ks':         A_Ks,
        'A_Ks_err':     A_Ks_err,
        'A_BPminRP':    A_V * A_BPminRP_div_AV,
        'dustmap_used': dustmap,
    })

print('Done.')

Querying dust maps for 112 stars...
Done.


In [9]:
ext_df = pd.DataFrame(ext_rows)
ext_df['source_id'] = ext_df['source_id'].astype('Int64')

for col in ['A_Ks','A_Ks_err','A_BPminRP','dustmap_used','bp_rp_0']:
    if col in src.columns:
        src = src.drop(columns=[col])

src = src.merge(ext_df, on='source_id', how='left')
src['A_Ks']     = src['A_Ks'].fillna(0.0)
src['A_Ks_err'] = src['A_Ks_err'].fillna(0.0)
src['A_BPminRP'] = src['A_BPminRP'].fillna(0.0)

# bp_rp_0: primary = Gaia GSP-Phot E(BP-RP); fallback = dust-map A_BPminRP
src['bp_rp_0'] = np.where(
    src['ebpminrp_gspphot'].notna(),
    src['bp_rp'] - src['ebpminrp_gspphot'],
    src['bp_rp'] - src['A_BPminRP'],
)

print('Extinction summary:')
print(src[['star_id','A_Ks','dustmap_used','bp_rp','ebpminrp_gspphot','bp_rp_0']].to_string())

Extinction summary:
                           star_id      A_Ks dustmap_used     bp_rp  ebpminrp_gspphot   bp_rp_0
0                      KIC 4914423  0.006647    Edenhofer  0.795113            0.0004  0.794713
1                      KIC 6116048  0.003020    Edenhofer  0.725575            0.0002  0.725375
2                      KIC 6116048  0.003020    Edenhofer  0.725575            0.0002  0.725375
3                      KIC 9410862  0.002413    Edenhofer  0.713387            0.0139  0.699487
4                      KIC 7106245  0.002954    Edenhofer  0.754281            0.0038  0.750481
5                      KIC 4914923  0.003459    Edenhofer  0.818376            0.0173  0.801076
6                      KIC 6933899  0.002937    Edenhofer  0.798972            0.0388  0.760172
7                      KIC 6521045  0.005959    Edenhofer  0.810572            0.0119  0.798672
8                      KIC 6521045  0.005956    Edenhofer  0.810572            0.0119  0.798672
9                   

## Step 4 — 2MASS crossmatch

Primary: ARI TAP `gaiadr3.tmass_psc_xsc_best_neighbour` → CDS TAPVizieR `II/246/out`.  
Fallback: SIMBAD `ids` field (reuses results from Step 1 where available, re-queries for EDR3/DR3 stars).

In [10]:
def query_2mass_designations_ari(source_ids, retries=3, delay=20):
    ids_str = ','.join(str(x) for x in source_ids)
    query = f"""
    SELECT source_id, original_ext_source_id AS twomass_id
    FROM gaiadr3.tmass_psc_xsc_best_neighbour
    WHERE source_id IN ({ids_str})
    """
    for attempt in range(1, retries + 1):
        try:
            r = requests.post(ARI_SYNC,
                data={'REQUEST':'doQuery','LANG':'ADQL','FORMAT':'json','QUERY':query},
                timeout=120)
            r.raise_for_status()
            d = r.json()
            cols = [c['name'] for c in d['metadata']]
            return pd.DataFrame(d['data'], columns=cols)
        except Exception as e:
            if attempt < retries:
                time.sleep(delay)
            else:
                return pd.DataFrame(columns=['source_id','twomass_id'])

def query_vizier_tmass(designations, retries=3, delay=20):
    desig_str = ','.join(f"'{d.strip()}'" for d in designations)
    adql = f"""
        SELECT "2MASS" AS twomass_id, "Kmag" AS k_m, "e_Kmag" AS k_cmsig
        FROM "II/246/out"
        WHERE "2MASS" IN ({desig_str})
    """
    for attempt in range(1, retries + 1):
        try:
            r = requests.post(CDS_TAP,
                data={'REQUEST':'doQuery','LANG':'ADQL','FORMAT':'json','QUERY':adql},
                timeout=120)
            r.raise_for_status()
            d = r.json()
            cols = [c['name'] for c in d['metadata']]
            df = pd.DataFrame(d['data'], columns=cols)
            df['twomass_id'] = df['twomass_id'].str.strip()
            return df
        except Exception as e:
            if attempt < retries:
                print(f'  VizieR attempt {attempt} failed ({e}), retrying in {delay}s...')
                time.sleep(delay)
            else:
                raise

print('2MASS helper functions defined.')

2MASS helper functions defined.


In [11]:
# Primary crossmatch via Gaia cross-match table
all_ids_list = src['source_id'].dropna().astype(int).unique().tolist()
print(f'Fetching 2MASS designations for {len(all_ids_list)} IDs from ARI...')

desig_parts = []
for i, chunk in enumerate(chunk_list(all_ids_list, n=500), start=1):
    print(f'  Chunk {i}: {len(chunk)} IDs', end=' ', flush=True)
    part = query_2mass_designations_ari(chunk)
    print(f'→ {len(part)} designations')
    desig_parts.append(part)
    time.sleep(0.5)

desig_df = pd.concat(desig_parts, ignore_index=True)
desig_df['source_id']  = desig_df['source_id'].astype('Int64')
desig_df['twomass_id'] = desig_df['twomass_id'].str.strip()
print(f'Total designations: {len(desig_df)}')

all_designations = desig_df['twomass_id'].dropna().unique().tolist()
print(f'Fetching Ks photometry for {len(all_designations)} designations from CDS...')
phot_df = query_vizier_tmass(all_designations)
phot_df['twomass_id'] = phot_df['twomass_id'].str.strip()
print(f'Photometry returned: {len(phot_df)}')

tmass_df = desig_df.merge(phot_df, on='twomass_id', how='left')

# Drop stale 2MASS columns if re-running
for col in ['twomass_id','k_m','k_cmsig']:
    if col in src.columns:
        src = src.drop(columns=[col])

src = src.merge(tmass_df[['source_id','twomass_id','k_m','k_cmsig']], on='source_id', how='left')
print(f'\n2MASS matched: {src["k_m"].notna().sum()}/{len(src)}')

Fetching 2MASS designations for 105 IDs from ARI...
  Chunk 1: 105 IDs → 77 designations
Total designations: 77
Fetching Ks photometry for 77 designations from CDS...
Photometry returned: 77

2MASS matched: 113/141


In [12]:
# SIMBAD fallback: for stars still missing k_m, query SIMBAD for 2MASS J designation.

def extract_2mass_from_simbad_ids(ids_str):
    if not ids_str or str(ids_str) == 'None':
        return None
    for ident in str(ids_str).split('|'):
        ident = ident.strip()
        if ident.startswith('2MASS J'):
            return ident[7:]
    return None

no_kmag = src[src['k_m'].isna() & src['source_id'].notna()].copy()
print(f'Stars without k_m: {len(no_kmag)}')

if len(no_kmag) > 0:
    # Build query names: for Gaia EDR3/DR3 stars, use 'Gaia DR3 {id}' to query SIMBAD
    query_names_fb = []
    for _, row in no_kmag.iterrows():
        name = str(row['star_id'])
        if re.match(r'Gaia (?:DR3|EDR3)', name):
            query_names_fb.append(f'Gaia DR3 {int(row["source_id"])}')
        else:
            query_names_fb.append(name)

    try:
        simbad_fb = Simbad()
        simbad_fb.add_votable_fields('ids')
        result_fb = simbad_fb.query_objects(query_names_fb)

        desig_map = {}
        for i, (idx, row) in enumerate(no_kmag.iterrows()):
            if i >= len(result_fb):
                break
            desig = extract_2mass_from_simbad_ids(result_fb['ids'][i] if result_fb is not None else None)
            if desig:
                desig_map[idx] = desig

        print(f'SIMBAD resolved 2MASS for {len(desig_map)}/{len(no_kmag)} unmatched stars')

        if desig_map:
            desigs_fb = list(set(desig_map.values()))
            phot_fb   = query_vizier_tmass(desigs_fb).set_index('twomass_id')
            filled = 0
            for idx, desig in desig_map.items():
                if desig in phot_fb.index and pd.notna(phot_fb.loc[desig, 'k_m']):
                    src.at[idx, 'twomass_id'] = desig
                    src.at[idx, 'k_m']        = float(phot_fb.loc[desig, 'k_m'])
                    src.at[idx, 'k_cmsig']    = (float(phot_fb.loc[desig, 'k_cmsig'])
                                                  if pd.notna(phot_fb.loc[desig, 'k_cmsig']) else 0.0)
                    filled += 1
            print(f'Filled {filled} rows via SIMBAD 2MASS fallback')
    except Exception as e:
        print(f'SIMBAD 2MASS fallback failed: {e}')

print(f'\n2MASS after fallback: {src["k_m"].notna().sum()}/{len(src)}')
print('Stars still missing k_m:')
print(src.loc[src['k_m'].isna(), ['star_id','source_id','parallax']].to_string())

Stars without k_m: 28
SIMBAD resolved 2MASS for 2/28 unmatched stars
Filled 2 rows via SIMBAD 2MASS fallback

2MASS after fallback: 115/141
Stars still missing k_m:
                           star_id            source_id  parallax
47   Gaia EDR3 2051099015605500672  2051099015605500672  0.036966
49   Gaia EDR3 2051104891118311040  2051104891118311040 -0.351026
50   Gaia EDR3 2051104891120796672  2051104891120796672 -0.020236
57   Gaia EDR3 2051105273377206784  2051105273377206784  0.046084
58   Gaia EDR3 2051105342091728384  2051105342091728384  0.278498
60   Gaia EDR3 2051105372154855552  2051105372154855552  0.189014
61   Gaia EDR3 2051105372157195264  2051105372157195264  0.432269
62   Gaia EDR3 2051105578318626304  2051105578318626304  0.157348
66   Gaia EDR3 2051105818833811584  2051105818833811584  0.077283
68   Gaia EDR3 2051105921913036160  2051105921913036160  0.552259
69   Gaia EDR3 2051105926212896512  2051105926212896512 -0.070417
70   Gaia EDR3 2051107055787471104  2051107

## Step 5 — Mann 2019 Mass-Ks relation

Valid range: **4.5 < M_Ks < 10.5** (0.075–0.70 M☉).

Pre-loading the FITS posterior for efficiency (same approach as `mass_recompute.ipynb`).

Asymmetric uncertainties: median − P16 (err_lo) and P84 − median (err_hi).

In [13]:
MANN_MKS_MIN = 4.5
MANN_MKS_MAX = 10.5

# Dereddened apparent mag
src['k_m_0'] = np.where(
    src['k_m'].notna(),
    src['k_m'] - src['A_Ks'].fillna(0.0),
    np.nan
)

# Absolute Ks using parallax distance
def row_MKs(row):
    if pd.isna(row['k_m_0']):
        return np.nan
    plx = row['parallax']
    if pd.isna(plx) or plx <= 0:
        return np.nan
    dist_pc = 1000.0 / plx
    return row['k_m_0'] + 5 - 5 * np.log10(dist_pc)

src['M_Ks'] = src.apply(row_MKs, axis=1)
src['flag_outside_mann_range'] = src['M_Ks'].apply(
    lambda m: (m < MANN_MKS_MIN or m > MANN_MKS_MAX) if pd.notna(m) else None
)

print('M_Ks per star:')
print(src[['star_id','k_m','k_m_0','M_Ks','flag_outside_mann_range']].to_string())
print()
in_range = src['flag_outside_mann_range'].eq(False).sum()
print(f'Stars in Mann range (4.5 < M_Ks < 10.5): {in_range}/{len(src)}')

M_Ks per star:
                           star_id     k_m      k_m_0      M_Ks flag_outside_mann_range
0                      KIC 4914423  10.873  10.866353  2.344412                    True
1                      KIC 6116048   7.121   7.117980  2.742904                    True
2                      KIC 6116048   7.121   7.117980  2.742904                    True
3                      KIC 9410862   9.375   9.372587  2.859090                    True
4                      KIC 7106245   9.419   9.416046  2.930537                    True
5                      KIC 4914923   7.935   7.931541  2.472551                    True
6                      KIC 6933899   8.171   8.168063  1.875746                    True
7                      KIC 6521045   9.768   9.762041  2.326682                    True
8                      KIC 6521045   9.768   9.762044  2.326685                    True
9                      KIC 3544595   8.370   8.367760  3.456017                    True
10               

In [14]:
# Pre-load Mann FITS posterior once
datapath  = pkg_resources.resource_filename('mk_mass', 'resources')
post_data = fits.open(datapath + '/Mk-M_7_trim.fits')[0].data
print(f'FITS posterior loaded: {post_data.shape[0]:,} samples × {post_data.shape[1]} parameters')

def compute_mass_row(row, post):
    """Return (median, err_lo, err_hi) from full Mann 2019 posterior."""
    if pd.isna(row.get('k_m_0')) or pd.isna(row.get('M_Ks')):
        return np.nan, np.nan, np.nan
    if row.get('flag_outside_mann_range'):   # catches True, np.bool_(True)
        return np.nan, np.nan, np.nan
    plx = row.get('parallax')
    plx_err = row.get('parallax_error')
    if pd.isna(plx) or plx <= 0:
        return np.nan, np.nan, np.nan
    dist  = 1000.0 / plx
    edist = (1000.0 / plx**2) * plx_err if pd.notna(plx_err) else 0.0
    ek_phot = row['k_cmsig'] if pd.notna(row.get('k_cmsig')) else 0.0
    ek_ext  = row['A_Ks_err'] if pd.notna(row.get('A_Ks_err')) else 0.0
    ek = np.sqrt(ek_phot**2 + ek_ext**2)
    try:
        samples = mk_mass.posterior(
            K=row['k_m_0'], dist=dist, ek=ek, edist=edist,
            oned=False, silent=True, post=post
        )
        samples = np.asarray(samples)
        valid   = samples[np.isfinite(samples)]
        med     = np.median(valid)
        p16     = np.percentile(valid, 16)
        p84     = np.percentile(valid, 84)
        return med, med - p16, p84 - med
    except Exception:
        return np.nan, np.nan, np.nan

print('Computing masses...')
mass_results = src.apply(lambda row: compute_mass_row(row, post_data), axis=1)
src['mass_msun']        = [r[0] for r in mass_results]
src['mass_msun_err_lo'] = [r[1] for r in mass_results]
src['mass_msun_err_hi'] = [r[2] for r in mass_results]

n_mass = src['mass_msun'].notna().sum()
print(f'Masses computed: {n_mass}/{len(src)}')
print()
print('M dwarfs in Mann range:')
print(src.loc[src['mass_msun'].notna(),
              ['star_id','spectral_type_estimate','mass_msun','mass_msun_err_lo','mass_msun_err_hi','M_Ks','prot_days']].to_string())

FITS posterior loaded: 400,000 samples × 7 parameters
Computing masses...
Masses computed: 10/141

M dwarfs in Mann range:
                          star_id spectral_type_estimate  mass_msun  mass_msun_err_lo  mass_msun_err_hi      M_Ks  prot_days
41                          GJ 84                    NaN   0.474442          0.011099          0.011212  5.814942      44.51
42                         GJ 176                    NaN   0.489648          0.012268          0.012467  5.721769      38.92
43                         GJ 205                    NaN   0.562776          0.042090          0.041437  5.258072      33.61
44                        GJ 618A                    NaN   0.394884          0.009265          0.009425  6.299664      56.52
45                         GJ 411                    NaN   0.367162          0.053556          0.057414  6.471597      48.00
46                         GJ 699                    NaN   0.160511          0.003835          0.003857  8.213841     130.00
10

## Step 6 — Gossage 2024 τ_cE and Rossby number

Same Table 2 and interpolation as `rossby_number.ipynb`. Only computed for stars with valid mass.

In [15]:
gossage_table2 = pd.DataFrame({
    'mass_msun': [0.18, 0.22, 0.30, 0.38, 0.43, 0.53, 0.60, 0.67, 0.73, 0.81, 0.89, 0.96, 1.01, 1.09],
    'tau_ce':    [214.59, 203.91, 239.77, 135.36, 81.54, 64.29, 43.61, 36.77, 32.49, 23.95, 19.49, 14.58, 10.96, 7.46],
    'tau_ce_err':[30.90, 38.73, 49.66, 10.04, 5.96, 4.71, 2.50, 1.79, 1.64, 1.26, 0.73, 0.53, 0.41, 0.42],
})

tau_ce_interp = interp1d(
    gossage_table2['mass_msun'], gossage_table2['tau_ce'],
    kind='linear', bounds_error=False,
    fill_value=(gossage_table2['tau_ce'].iloc[0], gossage_table2['tau_ce'].iloc[-1]),
)
tau_ce_err_interp = interp1d(
    gossage_table2['mass_msun'], gossage_table2['tau_ce_err'],
    kind='linear', bounds_error=False,
    fill_value=(gossage_table2['tau_ce_err'].iloc[0], gossage_table2['tau_ce_err'].iloc[-1]),
)

has_mass   = src['mass_msun'].notna()
mass_valid = src.loc[has_mass, 'mass_msun'].values   # plain numpy array, no alignment risk

src.loc[has_mass, 'tau_ce_days']     = tau_ce_interp(mass_valid)
src.loc[has_mass, 'tau_ce_err_days'] = tau_ce_err_interp(mass_valid)
src.loc[has_mass, 'rossby_number']   = src.loc[has_mass, 'prot_days'] / src.loc[has_mass, 'tau_ce_days']

print('τ_cE and Rossby number computed for M dwarfs in range:')
print(src.loc[has_mass, ['star_id','mass_msun','tau_ce_days','prot_days','rossby_number']].to_string())

τ_cE and Rossby number computed for M dwarfs in range:
                          star_id  mass_msun  tau_ce_days  prot_days  rossby_number
41                          GJ 84   0.474442    73.873700      44.51       0.602515
42                         GJ 176   0.489648    71.250805      38.92       0.546239
43                         GJ 205   0.562776    54.607101      33.61       0.615488
44                        GJ 618A   0.394884   119.338440      56.52       0.473611
45                         GJ 411   0.367162   152.115117      48.00       0.315550
46                         GJ 699   0.160511   214.590000     130.00       0.605806
109   Gaia DR3 861184515292492288   0.225437   206.347114      26.70       0.129394
110  Gaia DR3 1214994506568260608   0.348567   176.384093      46.60       0.264196
111  Gaia DR3 2816359731303136384   0.372379   145.307009      29.80       0.205083
112  Gaia DR3 3369544303487186560   0.220569   204.164992      13.40       0.065633


## Step 7 — Filter, map schema, stage for review

Tier 3 stars ("inspect before training") are **never auto-appended** — they are written to a
separate review file. Only Tier 1 / Tier 2 M dwarfs that pass the Mann range proceed to the staging step.

**Workflow:**
1. Run this cell → builds output rows, splits Tier 1/2 vs Tier 3
2. Run the staging cell → writes `cf_data/old_rotation_tiers_candidates.csv`
3. **Open and review that file** before continuing
4. Run the final append cell to commit to `training_stars.csv`

In [16]:
# Verified against ADS/arXiv
BIBCODE_TO_PAPER = {
    '2023arXiv230905666S': 'Saunders2023',    # Saunders et al. 2023, Kepler asteroseismic field dwarfs
    '2007AcA....57..149K': 'Kiraga2007',      # Kiraga & Stepien 2007, ASAS old-disk M dwarfs
    '2022AcA....72...77S': 'Sanjayan2022',    # Sanjayan et al. 2022, NGC 6791 rotational variables
    '2024ApJ...977...15C': 'Chiti2024',       # Chiti et al. 2024, WD+MS wide binaries
    '2016MNRAS.456..119C': 'Ceillier2016',    # Ceillier et al. 2016, Kepler surface rotation
    '2025ApJ...984..125L': 'Li2025',          # Li et al. 2025, HD 219134 asteroseismic K-dwarf
}

# binary_type based on sample_name
def get_binary_type(row):
    sname = str(row.get('sample_name', ''))
    if 'WD' in sname or 'white dwarf' in sname.lower():
        return 'WDM'
    return 'none'

print('bibcode → source_paper mapping:')
for bib, paper in BIBCODE_TO_PAPER.items():
    n = (src['bibcode'] == bib).sum()
    print(f'  {bib!r:40s} → {paper!r}  ({n} stars)')

bibcode → source_paper mapping:
  '2023arXiv230905666S'                    → 'Saunders2023'  (41 stars)
  '2007AcA....57..149K'                    → 'Kiraga2007'  (6 stars)
  '2022AcA....72...77S'                    → 'Sanjayan2022'  (62 stars)
  '2024ApJ...977...15C'                    → 'Chiti2024'  (5 stars)
  '2016MNRAS.456..119C'                    → 'Ceillier2016'  (26 stars)
  '2025ApJ...984..125L'                    → 'Li2025'  (1 stars)


In [17]:
valid_mask = src['mass_msun'].notna()
out = src[valid_mask].copy()

# Tier 3 stars require manual inspection — separate them out
tier3_mask = out['tier_name'].str.startswith('Tier 3', na=False)
out_t3  = out[tier3_mask].copy()
out_t12 = out[~tier3_mask].copy()

print(f'Valid M dwarfs in Mann range: {len(out)} total')
print(f'  Tier 1/2 (eligible for append): {len(out_t12)}')
print(f'  Tier 3   (review only):         {len(out_t3)}')
print()

train_schema = [
    'source_paper','star_name','source_id','ra','dec','pmra','pmdec',
    'prot_days','age_gyr','age_err_lo_gyr','age_err_hi_gyr','age_method',
    'parallax','parallax_error','bp_rp','bp_rp_0','ebpminrp_gspphot',
    'ruwe','high_ruwe','phot_g_mean_mag','binary_type',
    'twomass_id','k_m','k_cmsig','A_Ks','A_Ks_err','k_m_0','M_Ks',
    'flag_outside_mann_range','mass_msun','mass_msun_err_lo','mass_msun_err_hi',
    'tau_ce_days','rossby_number','tau_ce_err_days',
]

def build_output_rows(df):
    return pd.DataFrame({
        'source_paper':            df['bibcode'].map(BIBCODE_TO_PAPER).fillna(df['bibcode']),
        'star_name':               df['star_id'],
        'source_id':               df['source_id'].astype('Int64'),
        'ra':                      df['ra'],
        'dec':                     df['dec'],
        'pmra':                    df['pmra'],
        'pmdec':                   df['pmdec'],
        'prot_days':               df['prot_days'],
        'age_gyr':                 df['age_gyr'],
        'age_err_lo_gyr':          df['age_err_minus_gyr'],
        'age_err_hi_gyr':          df['age_err_plus_gyr'],
        'age_method':              df['age_label_type'],
        'parallax':                df['parallax'],
        'parallax_error':          df['parallax_error'],
        'bp_rp':                   df['bp_rp'],
        'bp_rp_0':                 df['bp_rp_0'],
        'ebpminrp_gspphot':        df['ebpminrp_gspphot'],
        'ruwe':                    df['ruwe'],
        'high_ruwe':               df['ruwe'] >= 1.2,
        'phot_g_mean_mag':         df['phot_g_mean_mag'],
        'binary_type':             df.apply(get_binary_type, axis=1),
        'twomass_id':              df['twomass_id'],
        'k_m':                     df['k_m'],
        'k_cmsig':                 df['k_cmsig'],
        'A_Ks':                    df['A_Ks'],
        'A_Ks_err':                df['A_Ks_err'],
        'k_m_0':                   df['k_m_0'],
        'M_Ks':                    df['M_Ks'],
        'flag_outside_mann_range': df['flag_outside_mann_range'],
        'mass_msun':               df['mass_msun'],
        'mass_msun_err_lo':        df['mass_msun_err_lo'],
        'mass_msun_err_hi':        df['mass_msun_err_hi'],
        'tau_ce_days':             df['tau_ce_days'],
        'rossby_number':           df['rossby_number'],
        'tau_ce_err_days':         df['tau_ce_err_days'],
    })

out_rows_t12 = build_output_rows(out_t12)
out_rows_t3  = build_output_rows(out_t3)

assert list(out_rows_t12.columns) == train_schema or len(out_rows_t12) == 0
assert list(out_rows_t3.columns)  == train_schema or len(out_rows_t3)  == 0
print('Schema check passed.')

Valid M dwarfs in Mann range: 10 total
  Tier 1/2 (eligible for append): 10
  Tier 3   (review only):         0

Schema check passed.


In [18]:
# Duplicate check against existing training_stars.csv
train = pd.read_csv(CF_DATA / 'training_stars.csv', dtype={'source_id': 'Int64'})
existing_ids = set(train['source_id'].dropna().astype(int))

def split_new_dups(rows_df, label):
    if len(rows_df) == 0:
        print(f'{label}: 0 candidates')
        return rows_df, rows_df.iloc[:0]
    ids = rows_df['source_id'].dropna().astype(int)
    new  = rows_df[~ids.isin(existing_ids)].copy()
    dups = rows_df[ ids.isin(existing_ids)].copy()
    print(f'{label}: {len(new)} new | {len(dups)} already in training_stars')
    for _, row in dups.iterrows():
        ex = train[train['source_id'] == row['source_id']]
        print(f'  SKIP {row["star_name"]} → already {ex["star_name"].iloc[0]} / {ex["source_paper"].iloc[0]}')
    return new, dups

new_t12, dups_t12 = split_new_dups(out_rows_t12, 'Tier 1/2')
new_t3,  dups_t3  = split_new_dups(out_rows_t3,  'Tier 3 (review)')

# Save staging file for Tier 1/2 candidates
candidates_path = CF_DATA / 'old_rotation_tiers_candidates.csv'
if len(new_t12) > 0:
    new_t12.to_csv(candidates_path, index=False)
    print(f'\n✓ Staged {len(new_t12)} rows → {candidates_path}')
    print('  Open cf_data/old_rotation_tiers_candidates.csv and verify before running the append cell.')
else:
    print('\nNo new Tier 1/2 rows to stage.')

# Save Tier 3 review file (separate, never auto-appended)
if len(out_rows_t3) > 0:
    t3_path = CF_DATA / 'old_rotation_tiers_tier3_review.csv'
    out_rows_t3.to_csv(t3_path, index=False)
    print(f'\n⚠  {len(out_rows_t3)} Tier 3 candidate(s) saved to {t3_path}')
    print('   These require manual inspection (binarity, blending, pulsation) before any use.')

Tier 1/2: 2 new | 8 already in training_stars
  SKIP GJ 84 → already GJ 84 / Kiraga2007
  SKIP GJ 176 → already GJ 176 / Kiraga2007
  SKIP GJ 205 → already GJ 205 / Kiraga2007
  SKIP GJ 618A → already GJ 618A / Kiraga2007
  SKIP GJ 411 → already GJ 411 / Kiraga2007
  SKIP GJ 699 → already GJ 699 / Kiraga2007
  SKIP Gaia DR3 861184515292492288 → already Gaia DR3 861184515292492288 / Chiti2024
  SKIP Gaia DR3 1214994506568260608 → already Gaia DR3 1214994506568260608 / Chiti2024
Tier 3 (review): 0 candidates

✓ Staged 2 rows → cf_data/old_rotation_tiers_candidates.csv
  Open cf_data/old_rotation_tiers_candidates.csv and verify before running the append cell.


In [19]:
# ─────────────────────────────────────────────────────────────────────────────
# STOP — review cf_data/old_rotation_tiers_candidates.csv before running this.
# ─────────────────────────────────────────────────────────────────────────────
#
# This cell reads the staged CSV back (so edits you make there are respected),
# checks it still matches the training_stars schema, then appends.

staged = pd.read_csv(candidates_path, dtype={'source_id': 'Int64'})
print(f'Staged rows loaded: {len(staged)}')
print(staged[['source_paper','star_name','mass_msun','prot_days','age_gyr','rossby_number']].to_string())
print()

if list(staged.columns) != train_schema:
    raise ValueError(f'Column mismatch — do not edit column headers in the candidates file.\n'
                     f'Expected: {train_schema}\nGot:      {list(staged.columns)}')

if len(staged) == 0:
    print('Nothing to append.')
else:
    train_final = pd.read_csv(CF_DATA / 'training_stars.csv', dtype={'source_id': 'Int64'})
    already_in  = staged['source_id'].astype('Int64').isin(train_final['source_id'].dropna())
    if already_in.any():
        print(f'WARNING: {already_in.sum()} row(s) in staged file are already in training_stars — skipping them.')
        staged = staged[~already_in]

    train_updated = pd.concat([train_final, staged[train_schema]], ignore_index=True)
    train_updated.to_csv(CF_DATA / 'training_stars.csv', index=False)
    print(f'Appended {len(staged)} rows. training_stars.csv: {len(train_final)} → {len(train_updated)} rows.')

Staged rows loaded: 2
  source_paper                     star_name  mass_msun  prot_days  age_gyr  rossby_number
0    Chiti2024  Gaia DR3 2816359731303136384   0.372379       29.8     7.46       0.205083
1    Chiti2024  Gaia DR3 3369544303487186560   0.220569       13.4     7.07       0.065633

Appended 2 rows. training_stars.csv: 6127 → 6129 rows.
